# Binary Neural Network

# Basic setting

### Measuring execution time

In [ ]:
!pip install ipython-autotime
%load_ext autotime

### Library

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix


from math import sqrt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

### Random seed

In [ ]:
SEED = 42

# Data collection

### Load data from url

In [ ]:
# Load the diabetes dataset from a GitHub repository
path = 'https://raw.githubusercontent.com/20161609/data_box/refs/heads/main/diabetes.csv'
df = pd.read_csv(path)

# Display the shape of the dataset (rows, columns)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Sort 'Pregnancies' column in descending order
tmp = df['Pregnancies'].sort_values(ascending=False)
tmp = tmp.reset_index()
tmp.head()

### Visualization

In [ ]:
# Visualize the distribution of the 'Outcome' column
sns.countplot(data=df, x='Outcome')

In [ ]:
# Visualize sorted 'Pregnancies' values
sns.barplot(x=tmp.index, y=tmp['Pregnancies'])

In [ ]:
# Generate histograms for all columns
df.hist()

# Data preprocessing

### Missing data - Null

In [ ]:
# Handle missing values in numeric columns by filling with mean
numeric_cols = df.select_dtypes(include=['number']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        print(f"Filling missing values in numeric column '{col}' with mean.")
        df[col].fillna(df[col].mean(), inplace=True)

# Handle missing values in categorical columns by filling with mode
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
numerical_cols = df.select_dtypes(include=['number']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        print(f"Filling missing values in categorical column '{col}' with mode.")
        df[col].fillna(df[col].mode()[0], inplace=True)


### Encode categorical variables

In [ ]:
# Convert categorical columns to numerical using Label Encoding
for col in categorical_cols:
  print(f"Encoding categorical column '{col}'.")

  le = LabelEncoder()
  # Convert to string before encoding
  df[col] = le.fit_transform(df[col].astype(str))

print("Missing values after preprocessing:")
print(df.isnull().sum())

### Duplication

In [ ]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
final_rows = df.shape[0]
print(f"Removed {initial_rows - final_rows} duplicate rows.")

# Data preparation

### Split: train and test

In [ ]:
train, test = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df['Outcome'])
train.shape, test.shape

### split: features and target

In [ ]:
X_train = train.drop('Outcome', axis=1)
y_train = train['Outcome']

X_train.shape, y_train.shape

### Replace zero values in specific columns with the median

In [ ]:
median_list = []

col_list = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in col_list:
  med = X_train[col].median()
  X_train.loc[X_train[col] == 0, col] = med
  median_list.append(med)

median_list

### Scaling

In [ ]:
ss = StandardScaler()
X_train_s = ss.fit_transform(X_train)  # Convert to numpy array automatically

X_train_s.shape

In [ ]:
# Convert target values to numpy array
y_train_e = y_train.to_numpy()
y_train_e.shape

# Model

### Create model

In [ ]:
model = keras.Sequential([
    layers.Dense(8, activation='relu', input_shape=(8,)),  # Input layer
    layers.Dense(8, activation='relu'),                   # Hidden layer
    layers.Dense(8, activation='relu'),                   # Hidden layer
    layers.Dense(1, activation='sigmoid')                 # Output layer (sigmoid for binary classification)
])

model.summary()

### Compile model

In [ ]:
# Compile the model with loss function, optimizer, and metrics
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

### History

In [ ]:
# Define training parameters
EPOCHS = 100
BATCH_SIZE = 16

# Train the model
history = model.fit(
  X_train_s, y_train_e,
  epochs=EPOCHS,
  batch_size=BATCH_SIZE,
  validation_split=0.2
)


### Plot history

In [ ]:
# Plot training history (loss and accuracy curves)
def plot_history(history):
  hist = pd.DataFrame(history.history)
  hist['epoch'] = history.epoch
  plt.figure(figsize=(16, 8))

  # Plot training and validation loss
  plt.subplot(1, 2, 1)
  plt.xlabel('epochs')
  plt.ylabel('loss')
  plt.plot(hist['epoch'], hist['loss'], label='train loss')
  plt.plot(hist['epoch'], hist['val_loss'], label='val loss')
  plt.title('Loss Curve')
  plt.legend()

  # Plot training and validation accuracy
  plt.subplot(1, 2, 2)
  plt.xlabel('epochs')
  plt.ylabel('accuracy')
  plt.plot(hist['epoch'], hist['accuracy'], label='train accuracy')
  plt.plot(hist['epoch'], hist['val_accuracy'], label='val accuracy')
  plt.title('Accuracy Curve')
  plt.legend()
  plt.show()

# Call the function to visualize training progress
plot_history(history)

# Evaluate

In [ ]:
# Prepare the test dataset
X_test = test.drop('Outcome', axis=1)
y_test = test['Outcome']

X_test.shape, y_test.shape

In [ ]:
# Replace zero values in the test dataset with the corresponding medians from training
for i, col in enumerate(col_list):
    X_test.loc[X_test[col] == 0, col] = median_list[i]

# Standardize the test data using the same scaler
X_test_s = ss.transform(X_test)

# Convert test target values to numpy array
y_test_e = y_test.to_numpy()

### Prediction

In [ ]:
# Predict on the test data
y_pred = model.predict(X_test_s)
y_pred = (y_pred > 0.5).astype(int).reshape(-1)  # Convert probabilities to binary predictions

y_pred.shape

In [ ]:
def print_metrics(y_true, y_pred, ave='binary'):
    # Print evaluation metrics
    print('accuracy:', accuracy_score(y_test_e, y_pred))
    print('recall:', recall_score(y_test_e, y_pred, average=ave))
    print('precision:', precision_score(y_test_e, y_pred, average=ave))
    print('f1 :', f1_score(y_test_e, y_pred, average=ave))

    # Display confusion matrix as a heatmap
    clm = confusion_matrix(y_test_e, y_pred)
    s = sns.heatmap(clm, annot=True, fmt='d', cbar=False)
    s.set(xlabel='Predicted', ylabel='Actual')
    plt.show()

# Call the function to display metrics and confusion matrix
print_metrics(y_test_e, y_pred)
